在Langchain中，发送给模型的消息、模型返回的消息都会统一被封装为BaseMessage，并且准备了多个BaseMessage的子类对应不同角色类型的消息

- SystemMessage: role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage: role是user，代表用户输入的消息
- AIMessage: role是assistant，代表LLM生成的响应，包含：文本、工具调用、源数据
- ToolMessage: role是tool，代表工具调用时产生的结果

In [1]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordiantes
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-v4-flash", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        # {"role": "system", "content": "你是一个热心的AI助手"},
        # {"role": "user", "content": "你好，我是Orien"},
        # {"role": "assistant", "content": "你好，Orien，很高兴认识你"},
        # {"role": "user", "content": "今天佛山的天气如何？"},
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是Orien"),
        AIMessage("你好，Orien，很高兴认识你"),
        HumanMessage("今天佛山的天气如何？")
    ]
})

print(response)

{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='5b472316-b5af-4312-b07a-e705416d1e77'), HumanMessage(content='你好，我是Orien', additional_kwargs={}, response_metadata={}, id='e5f70561-00a8-4583-9218-44b04bb8046e'), AIMessage(content='你好，Orien，很高兴认识你', additional_kwargs={}, response_metadata={}, id='d0432a1e-ca8e-4d89-b582-468bd441f337', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='今天佛山的天气如何？', additional_kwargs={}, response_metadata={}, id='0dfbf5ab-f307-4a6e-9084-93034b9029d2'), AIMessage(content='好的，让我查一下今天佛山的天气情况。', additional_kwargs={'refusal': None, 'reasoning_content': '用户想知道佛山的天气情况。我可以使用get_weather工具来获取佛山的信息。'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 316, 'total_tokens': 389, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 18, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_w

In [2]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是Orien
================================== Ai Message ==================================

你好，Orien，很高兴认识你
================================ Human Message =================================

今天佛山的天气如何？
================================== Ai Message ==================================

好的，让我查一下今天佛山的天气情况。
Tool Calls:
  get_weather (call_00_rASfvJ7Zq45SmxFTloD03592)
 Call ID: call_00_rASfvJ7Zq45SmxFTloD03592
  Args:
    location: 佛山
================================= Tool Message =================================
Name: get_weather

Current weather in 佛山 is sunny
================================== Ai Message ==================================

今天佛山的天气是**晴朗（sunny）**的！☀️

是个好天气，适合出门活动。不过具体温度等信息目前没有详细数据，建议你出门前也可以查看一下实时温度，合理搭配衣物哦～ 

有什么其他需要帮忙的吗？😊


## 2.多模态消息  
前提是必须是多模态模型才支持

### 2.1在线图片

In [ ]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url = "",
    api_key = ""
)

# 创建agent
agent = create_agent(model=model)

In [ ]:
# 准备多模态消息
# message = {
#     "role": "user",
#     "content": [
#         {"type": "text", "text": "描述以下这张图片的内容"},
#         {"type": "image", "url": "在线图片的url"}
#     ]
# }

message = HumanMessage([
    {"type": "text", "text": "描述以下这张图片的内容"},
    {"type": "image", "url": "在线图片的url"}
])

In [ ]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)

for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

## 2.2.本地图片数据
当用户上传图片数据，而不是图片的url地址，我们需要将图片数据转换成base64字符串，然后发送给模型。

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [ ]:
# 读取图片，转为base64字符串
import base64

# 获取第一个上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,  # 传入base64编码
        "mime_type": "image/jpeg",  #告诉模型图片的形式
    },
    {"type": "text", "text": ""}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)